# K-Means Init vs Trained MFA: Assignment and Intrinsic-Dimension Comparison

MFA training in this repo initializes component means from reservoir-KMeans
centroids. This notebook quantifies **what training changes relative to that
initialization** for a single configuration (default: K=1000, layer 5, q=10),
by comparing two hard partitions of the *same* activation token stream:

- **k-means (init)**: each token assigned to the nearest Euclidean centroid
  (`kmeans_centroid_assignments.pt`, produced by
  `dalg-run-metrics assignments --medoids-path ...`)
- **MFA (trained)**: each token assigned to the argmax-responsibility component
  (`mfa_model_assignments.pt`, produced by
  `dalg-run-metrics assignments --data-dir ...`)

Because the MFA components are initialized *from these exact centroids*,
cluster id `k` of the k-means partition corresponds directly to MFA component
`k` — the notebook verifies this (Section 1.1) and then uses the **identity
mapping** for all per-cluster comparisons.

Sections:

1. Setup and artifact validation
2. Assignment agreement (global metrics + per-cluster Jaccard)
3. Centroid movement: how far the trained means moved from their init centroids
4. Intrinsic dimension: distributions and per-cluster change init → trained
5. Massive activations: rogue-dimension profile of centroids and means
6. Spectral profiles: full variance spectra of the two partitions
7. Learned subspaces: what span(W_k) aligns with
8. Compact conclusion

The notebook is safe to run with missing artifacts: dependent sections report
what is missing and skip.

## 1. Setup and Artifact Validation

In [1]:
from __future__ import annotations

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from scipy.optimize import linear_sum_assignment

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception as exc:
    px = None
    go = None
    print(f"Plotly unavailable: {exc}")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/dalg").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO = find_repo_root()

# ---------------------------------------------------------------- parameters
LAYER = 5
K = 1000
Q = 10  # MFA rank; only used to locate the MFA run and its intrinsic dims
EPOCHS = 1
MFA_ASSIGN="ar" # ar - nn: nearest neighbor assignment, ar: argmax responsibility
# ------------------------------------------------- paths built from parameters
MFA_RUN = REPO / f"dalg-cache/pile_gemma2b_models/layer{LAYER:02d}_{K}_{Q}_component_sharded_mfa"
# MFA_RUN = REPO / "dalg-cache/pile_gemma2b_models/layer05_1000_10_mfa_1epoch_20260703_1538"
_MFA_ASSIGN_FILES = {"nn": "mfa_model_nearest_centroid_assignments.pt", "ar": "mfa_model_assignments.pt"}
MFA_ASSIGN_PATH = MFA_RUN / _MFA_ASSIGN_FILES[MFA_ASSIGN]
MFA_INIT_CENTROIDS = MFA_RUN / "centroids.pt"

CENTROIDS_DIR = REPO / f"dalg-cache/pile_gemma2b_models/centroids/k{K}_L{LAYER:02d}"
KMEANS_CENTROIDS = CENTROIDS_DIR / "centroids.pt"
KMEANS_ASSIGN = CENTROIDS_DIR / "kmeans_centroid_assignments.pt"

MFA_ID_PATH = MFA_RUN / "intrinsic_dims.pt"
KMEANS_ID_PATH = REPO / f"dalg-cache/output/experiments/centroids_{K}_{LAYER:02d}/intrinsic_dims.pt"
MFA_TOP_PCS_PATH = MFA_RUN / "cluster_top_pcs.pt"
KMEANS_TOP_PCS_PATH = CENTROIDS_DIR / "cluster_top_pcs.pt"

# ------------------------------------------------------------- plot constants
COLOR_KMEANS = "#2a78d6"  # blue  — k-means init partition
COLOR_MFA = "#1baf7a"     # aqua  — trained MFA partition
COLOR_NEUTRAL = "#52514e" # gray  — derived quantities (deltas, joint scatters)
PLOT_TEMPLATE = "plotly_white"

# ------------------------------------------------------------- plot saving
PLOTS_DIR = REPO / "notebooks/plots"
PLOTS_HTML_DIR = PLOTS_DIR / "html"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_HTML_DIR.mkdir(parents=True, exist_ok=True)
PLOT_TAG = f"L{LAYER:02d}_K{K}_q{Q}_ep{EPOCHS}_{MFA_ASSIGN}"

def save_fig(fig, name: str) -> None:
    fig.update_layout(title_font_size=13, title_x=0.5, margin=dict(t=48))
    export_width = int(fig.layout.width or 1000)
    export_height = int(fig.layout.height or 550)
    path = PLOTS_DIR / f"{name}_{PLOT_TAG}.pdf"
    try:
        fig.write_image(str(path), width=export_width, height=export_height)
    except Exception as exc:
        path = PLOTS_HTML_DIR / f"{name}_{PLOT_TAG}.html"
        fig.write_html(str(path), include_plotlyjs="cdn")
        print(f"PDF export failed ({exc}); saved {path.name} instead")

artifacts = pd.DataFrame(
    [
        {"artifact": "kmeans assignments", "path": str(KMEANS_ASSIGN), "exists": KMEANS_ASSIGN.exists()},
        {"artifact": f"mfa assignments ({MFA_ASSIGN})", "path": str(MFA_ASSIGN_PATH), "exists": MFA_ASSIGN_PATH.exists()},
        {"artifact": "kmeans centroids", "path": str(KMEANS_CENTROIDS), "exists": KMEANS_CENTROIDS.exists()},
        {"artifact": "mfa init centroids", "path": str(MFA_INIT_CENTROIDS), "exists": MFA_INIT_CENTROIDS.exists()},
        {"artifact": "kmeans intrinsic dims", "path": str(KMEANS_ID_PATH), "exists": KMEANS_ID_PATH.exists()},
        {"artifact": "mfa intrinsic dims", "path": str(MFA_ID_PATH), "exists": MFA_ID_PATH.exists()},
        {"artifact": "kmeans top PCs", "path": str(KMEANS_TOP_PCS_PATH), "exists": KMEANS_TOP_PCS_PATH.exists()},
        {"artifact": "mfa top PCs", "path": str(MFA_TOP_PCS_PATH), "exists": MFA_TOP_PCS_PATH.exists()},
        {"artifact": "mfa model", "path": str(MFA_RUN / "mfa_model.pt"), "exists": (MFA_RUN / "mfa_model.pt").exists() or (MFA_RUN / "mfa_model_shards.json").exists()},
    ]
)
display(artifacts)

HAVE_ASSIGNMENTS = KMEANS_ASSIGN.exists() and MFA_ASSIGN_PATH.exists()
HAVE_IDS = KMEANS_ID_PATH.exists() and MFA_ID_PATH.exists()
HAVE_TOP_PCS = KMEANS_TOP_PCS_PATH.exists() and MFA_TOP_PCS_PATH.exists()
HAVE_MODEL = (
    (MFA_RUN / "mfa_model.pt").exists() or (MFA_RUN / "mfa_model_shards.json").exists()
) and KMEANS_CENTROIDS.exists()

,artifact,path,exists
0,kmeans assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
1,mfa assignments (ar),/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
2,kmeans centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
3,mfa init centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
4,kmeans intrinsic dims,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
5,mfa intrinsic dims,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
6,kmeans top PCs,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
7,mfa top PCs,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
8,mfa model,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True


In [2]:
# Shared PCA inputs for Sections 7 and 8. Keep the sidecars memory-mapped:
# together they contain roughly 1.6 GB of float32 directions.
top_pcs_km_obj = None
top_pcs_mfa_obj = None
pcs_km = None
pcs_mfa = None
valid_pcs_km = np.zeros(K, dtype=bool)
valid_pcs_mfa = np.zeros(K, dtype=bool)
valid_pcs_both = np.zeros(K, dtype=bool)

if not HAVE_TOP_PCS:
    display(Markdown("**Sections 7–8 unavailable:** missing one or both `cluster_top_pcs.pt` artifacts."))
else:
    top_pcs_km_obj = torch.load(
        KMEANS_TOP_PCS_PATH, map_location="cpu", weights_only=True, mmap=True
    )
    top_pcs_mfa_obj = torch.load(
        MFA_TOP_PCS_PATH, map_location="cpu", weights_only=True, mmap=True
    )

    assert int(top_pcs_km_obj["K"]) == int(top_pcs_mfa_obj["K"]) == K
    assert int(top_pcs_km_obj["top_pcs"]) == int(top_pcs_mfa_obj["top_pcs"])
    assert int(top_pcs_km_obj["max_samples"]) == int(top_pcs_mfa_obj["max_samples"])

    TOP_PCS = int(top_pcs_km_obj["top_pcs"])
    PCA_MAX_SAMPLES = int(top_pcs_km_obj["max_samples"])
    pcs_km = top_pcs_km_obj["cluster_top_pcs"]
    pcs_mfa = top_pcs_mfa_obj["cluster_top_pcs"]
    assert len(pcs_km) == len(pcs_mfa) == K

    nonempty_bases = [basis for basis in (*pcs_km, *pcs_mfa) if basis is not None]
    pca_dims = {int(basis.shape[1]) for basis in nonempty_bases}
    assert len(pca_dims) == 1, f"inconsistent PCA ambient dimensions: {pca_dims}"
    PCA_D = pca_dims.pop()

    for partition, bases in [("kmeans", pcs_km), ("mfa", pcs_mfa)]:
        for cluster_id, basis in enumerate(bases):
            if basis is None:
                continue
            assert basis.ndim == 2, (partition, cluster_id, basis.shape)
            assert 0 < basis.shape[0] <= TOP_PCS, (partition, cluster_id, basis.shape)
            assert basis.shape[1] == PCA_D, (partition, cluster_id, basis.shape)

    valid_pcs_km = np.asarray([basis is not None for basis in pcs_km])
    valid_pcs_mfa = np.asarray([basis is not None for basis in pcs_mfa])
    valid_pcs_both = valid_pcs_km & valid_pcs_mfa

    display(
        pd.DataFrame(
            [
                {
                    "partition": "kmeans (init)",
                    "valid clusters": int(valid_pcs_km.sum()),
                    "requested PCs": TOP_PCS,
                    "ambient dimension": PCA_D,
                    "max samples / cluster": PCA_MAX_SAMPLES,
                },
                {
                    "partition": "mfa (trained)",
                    "valid clusters": int(valid_pcs_mfa.sum()),
                    "requested PCs": TOP_PCS,
                    "ambient dimension": PCA_D,
                    "max samples / cluster": PCA_MAX_SAMPLES,
                },
            ]
        )
    )
    print(f"clusters with PCs in both partitions: {valid_pcs_both.sum()}/{K}")


,partition,valid clusters,requested PCs,ambient dimension,max samples / cluster
0,kmeans (init),985,100,2048,10000
1,mfa (trained),987,100,2048,10000


clusters with PCs in both partitions: 979/1000


### 1.1 Cluster Correspondence Check

The per-cluster comparisons below rely on cluster id `k` meaning the same thing
in both partitions. That holds if (and only if) the centroids used for the
nearest-centroid assignments are the same tensors the MFA was initialized from.
We check bit-identity; if it fails, fall back to Hungarian matching before
trusting any per-cluster plot.

In [3]:
IDENTITY_OK = False
if KMEANS_CENTROIDS.exists() and MFA_INIT_CENTROIDS.exists():
    def _centroid_tensor(path: Path) -> torch.Tensor:
        obj = torch.load(path, map_location="cpu")
        if isinstance(obj, dict):
            for key in ("centroids", "mu", "means"):
                if key in obj:
                    obj = obj[key]
                    break
        return obj.float()

    c_km = _centroid_tensor(KMEANS_CENTROIDS)
    c_init = _centroid_tensor(MFA_INIT_CENTROIDS)
    IDENTITY_OK = c_km.shape == c_init.shape and torch.equal(c_km, c_init)
    print(f"kmeans centroids: {tuple(c_km.shape)}, mfa init centroids: {tuple(c_init.shape)}")
    print(f"bit-identical: {IDENTITY_OK}")
    if not IDENTITY_OK:
        print(
            "WARNING: centroids differ -> cluster ids do NOT correspond; "
            "per-cluster sections would need Hungarian matching instead of identity."
        )
    del c_km, c_init
else:
    print("Skipped: missing one or both centroid files; assuming identity correspondence is UNVERIFIED.")

kmeans centroids: (1000, 2048), mfa init centroids: (1000, 2048)
bit-identical: True


## 2. Assignment Agreement

**Experiment.** Both assignment files label every token of the same activation
stream (layer `LAYER`, all shards, prefix-dropped) with one of the same K
cluster ids. The k-means partition is a Voronoi partition around the init
centroids; the MFA partition is the argmax of trained responsibilities, which
moves boundaries according to the learned local subspaces, noise, and mixture
weights. We measure how far training moved tokens across region boundaries:

- **Same-id agreement**: fraction of tokens whose cluster id is unchanged
  (valid because of the identity correspondence verified above).
- **NMI** (normalized mutual information, geometric-mean normalization):
  agreement up to a relabeling — high NMI with lower same-id agreement would
  mean training permuted regions rather than reshaping them.
- **Hungarian sanity check**: the optimal one-to-one matching between the two
  partitions should mostly be the identity if clusters stayed put.

All statistics derive from the K×K contingency matrix
`C[i, j] = #tokens with kmeans id i and mfa id j`, computed in one
`bincount` pass so the two ~N-length assignment vectors can be freed
immediately.

**Assignment variant.** `MFA_ASSIGN` in Section 1 selects which MFA-side
partition is compared: `"ar"` = argmax responsibility (the full likelihood
rule), `"nn"` = nearest Euclidean centroid *on the trained means*. Comparing
the two runs of this notebook decomposes the disagreement into "the means
moved" (`nn` vs k-means) and "the metric changed" (`ar` vs `nn`).

In [4]:
# ---- reusable helpers for the assignment-agreement analysis (Sections 2 and 2bis) ----
def load_assignments(path: Path) -> dict:
    obj = torch.load(path, map_location="cpu")
    obj["assignments"] = obj["assignments"].to(torch.long)
    obj["cluster_sizes"] = obj["cluster_sizes"].to(torch.long)
    return obj


def contingency_agreement(a_x: torch.Tensor, a_y: torch.Tensor, K: int) -> dict:
    """Same-id / NMI / Hungarian agreement from the K x K contingency of two labelings.

    ``C[i, j]`` counts tokens with label ``i`` under ``a_x`` and ``j`` under ``a_y``,
    accumulated in one ``bincount`` pass. NMI uses geometric-mean normalisation.
    """
    N = a_x.numel()
    contingency = (
        torch.bincount(a_x * K + a_y, minlength=K * K).reshape(K, K).numpy().astype(np.int64)
    )
    x_sizes = contingency.sum(axis=1)
    y_sizes = contingency.sum(axis=0)

    same_id = contingency.trace() / N

    def entropy_from_counts(counts: np.ndarray) -> float:
        p = counts[counts > 0].astype(np.float64) / counts.sum()
        return float(-(p * np.log(p)).sum())

    H_x = entropy_from_counts(x_sizes)
    H_y = entropy_from_counts(y_sizes)
    nz_i, nz_j = np.nonzero(contingency)
    n_ij = contingency[nz_i, nz_j].astype(np.float64)
    p_ij = n_ij / N
    p_i = x_sizes[nz_i].astype(np.float64) / N
    p_j = y_sizes[nz_j].astype(np.float64) / N
    MI = float((p_ij * np.log(p_ij / (p_i * p_j))).sum())
    NMI = MI / np.sqrt(H_x * H_y) if H_x > 0 and H_y > 0 else float("nan")

    row_ind, col_ind = linear_sum_assignment(-contingency)
    hungarian = contingency[row_ind, col_ind].sum() / N
    frac_self = float((row_ind == col_ind).mean())
    return {
        "N": N, "contingency": contingency, "x_sizes": x_sizes, "y_sizes": y_sizes,
        "same_id": same_id, "NMI": NMI, "hungarian": hungarian, "frac_self": frac_self,
    }


def global_metrics_table(stats: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"metric": "tokens (N)", "value": f"{stats['N']:,}"},
            {"metric": "same-id agreement", "value": f"{stats['same_id']:.2%}"},
            {"metric": "NMI", "value": f"{stats['NMI']:.2%}"},
            {"metric": "Hungarian matched agreement", "value": f"{stats['hungarian']:.2%}"},
            {"metric": "fraction of clusters self-matched (Hungarian)", "value": f"{stats['frac_self']:.1%}"},
        ]
    )



In [5]:
if not HAVE_ASSIGNMENTS:
    display(Markdown("**Skipped:** missing one or both assignment artifacts."))
else:
    km_obj = load_assignments(KMEANS_ASSIGN)
    mfa_obj = load_assignments(MFA_ASSIGN_PATH)
    a_km = km_obj["assignments"]
    a_mfa = mfa_obj["assignments"]
    assert int(km_obj["K"]) == int(mfa_obj["K"]) == K, (km_obj["K"], mfa_obj["K"], K)
    assert a_km.numel() == a_mfa.numel(), (a_km.numel(), a_mfa.numel())
    N = a_km.numel()
    print(f"Loaded {N:,} token assignments for both partitions")

    stats = contingency_agreement(a_km, a_mfa, K)
    contingency = stats["contingency"]
    km_sizes = stats["x_sizes"]
    mfa_sizes = stats["y_sizes"]
    same_id_agreement = stats["same_id"]
    NMI = stats["NMI"]
    frac_self_matched = stats["frac_self"]
    _same_cluster = np.diag(contingency)
    _cluster_union = km_sizes + mfa_sizes - _same_cluster
    per_cluster = pd.DataFrame(
        {
            "cluster": np.arange(K),
            "kmeans_size": km_sizes,
            "mfa_size": mfa_sizes,
            "intersection": _same_cluster,
            "jaccard": np.divide(
                _same_cluster, _cluster_union,
                out=np.zeros(K, dtype=float), where=_cluster_union > 0,
            ),
        }
    )
    del a_km, a_mfa, km_obj["assignments"], mfa_obj["assignments"]
    gc.collect()

    global_metrics = global_metrics_table(stats)
    display(global_metrics)


Loaded 73,687,936 token assignments for both partitions


,metric,value
0,tokens (N),"73,687,936"
1,same-id agreement,57.30%
2,NMI,72.39%
3,Hungarian matched agreement,57.35%
4,fraction of clusters self-matched (Hungarian),99.2%


## 2.1 Responsibility vs Nearest-Centroid MFA Assignments

Section 2 compared k-means against one MFA partition (`MFA_ASSIGN`). This section runs the
**same agreement analysis** between the two ways of turning the *trained* MFA into hard
labels:

- **`ar`** — argmax responsibility, the full likelihood rule (uses each component's mean,
  loadings `W_k`, and noise).
- **`nn`** — nearest Euclidean centroid on the trained means (means only).

Their disagreement isolates "the metric changed": where the learned local subspaces and
noise pull a token to a different component than the plain nearest mean would. High same-id
agreement means the two rules mostly coincide; a gap that NMI does not recover would mean
responsibility systematically reshapes regions rather than relabelling them.


In [6]:
AR_PATH = MFA_RUN / _MFA_ASSIGN_FILES["ar"]
NN_PATH = MFA_RUN / _MFA_ASSIGN_FILES["nn"]
HAVE_AR_NN = AR_PATH.exists() and NN_PATH.exists()

if not HAVE_AR_NN:
    display(Markdown("**Skipped:** requires both the `ar` and `nn` MFA assignment artifacts."))
else:
    a_ar = load_assignments(AR_PATH)["assignments"]
    a_nn = load_assignments(NN_PATH)["assignments"]
    assert a_ar.numel() == a_nn.numel(), (a_ar.numel(), a_nn.numel())
    print(f"Loaded {a_ar.numel():,} token assignments for both MFA criteria (ar, nn)")

    arnn_stats = contingency_agreement(a_ar, a_nn, K)
    display(global_metrics_table(arnn_stats))

    del a_ar, a_nn
    gc.collect()


Loaded 73,687,936 token assignments for both MFA criteria (ar, nn)


,metric,value
0,tokens (N),"73,687,936"
1,same-id agreement,75.08%
2,NMI,81.12%
3,Hungarian matched agreement,75.08%
4,fraction of clusters self-matched (Hungarian),100.0%


## 3. Centroid Movement: Init → Trained Means


**Experiment.** Each MFA mean `mu_k` starts at init centroid `c_k` (verified in
Section 1.1) and is free to move during training. We measure how far each one
went:

- **Displacement** `‖mu_k − c_k‖`, shown raw against a baseline of the average
  pairwise distance *between* centroids (within the k-means set and within the
  MFA means) — movement is only "small" if it is small relative to how far
  centroids are from each other; and a scale-free version dividing by the
  distance from `c_k` to its *nearest other* init centroid — a relative
  displacement > 1 means the mean moved farther than the local inter-centroid
  spacing, i.e. well out of its original Voronoi cell.
- **Cosine similarity** between `mu_k` and `c_k`, in two versions. The **raw**
  cosine is inflated: all activations share a massive offset concentrated in a
  couple of rogue dimensions (the global centroid mean has norm ≈ the centroid
  norms themselves), so even unrelated centroids score ≈0.95. The **centered**
  cosine removes the global mean `m` of the init centroids first,
  `cos(mu_k − m, c_k − m)`, and is the informative one: its cross-centroid
  baseline sits near 0. Both plots carry their own baselines (average pairwise
  cosine between *different* centroids within each set).
- **Nearest-init retention**: is `mu_k` still closer to `c_k` than to any other
  init centroid? Together with the Hungarian check of Section 2, this separates
  "means drifted locally" from "means migrated to other regions".
- **Movement vs membership change**: do clusters whose mean moved most also
  have the lowest assignment Jaccard (Section 2.1)?
- **Neighbour overlap of the centroid clouds** (3.1): for each cluster, how much
  of its k-NN set *among the other centroids* is shared between the init cloud
  and the trained-means cloud — did training preserve the relational layout of
  the centroids, or just their identities?


In [7]:
if not HAVE_MODEL:
    display(Markdown("**Skipped:** missing MFA model files or k-means centroids."))
else:
    from scipy.spatial.distance import cdist

    from dalg.models.mfa import load_mfa

    model = load_mfa(MFA_RUN / "mfa_model.pt", map_location="cpu")
    mu = model.mu.detach().float().cpu().numpy()
    del model
    gc.collect()

    c_init = _centroid_tensor(KMEANS_CENTROIDS).numpy()
    assert mu.shape == c_init.shape, (mu.shape, c_init.shape)

    displacement = np.linalg.norm(mu - c_init, axis=1)

    offdiag = ~np.eye(K, dtype=bool)
    init_pairwise = cdist(c_init, c_init)
    mu_pairwise = cdist(mu, mu)
    avg_dist_km = float(init_pairwise[offdiag].mean())
    avg_dist_mfa = float(mu_pairwise[offdiag].mean())

    np.fill_diagonal(init_pairwise, np.inf)
    nn_init_dist = init_pairwise.min(axis=1)
    rel_displacement = displacement / nn_init_dist

    def _unit(x: np.ndarray) -> np.ndarray:
        return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

    c_unit = _unit(c_init)
    mu_unit = _unit(mu)
    cos_sim = (mu_unit * c_unit).sum(axis=1)
    avg_cos_km = float((c_unit @ c_unit.T)[offdiag].mean())
    avg_cos_mfa = float((mu_unit @ mu_unit.T)[offdiag].mean())

    # raw activations share a massive offset (dominated by a couple of rogue
    # dimensions), which pushes ALL raw cosines toward 1; center by the global
    # mean of the init centroids to expose the informative directions
    global_mean = c_init.mean(axis=0)
    c_cent_unit = _unit(c_init - global_mean)
    mu_cent_unit = _unit(mu - global_mean)
    cos_sim_centered = (mu_cent_unit * c_cent_unit).sum(axis=1)
    avg_cos_km_centered = float((c_cent_unit @ c_cent_unit.T)[offdiag].mean())
    avg_cos_mfa_centered = float((mu_cent_unit @ mu_cent_unit.T)[offdiag].mean())

    nearest_init = cdist(mu, c_init).argmin(axis=1)
    frac_nearest_own_init = float((nearest_init == np.arange(K)).mean())

    move_df = pd.DataFrame(
        {
            "cluster": np.arange(K),
            "displacement": displacement,
            "nn_init_dist": nn_init_dist,
            "rel_displacement": rel_displacement,
            "cos_sim": cos_sim,
            "cos_sim_centered": cos_sim_centered,
            "nearest_own_init": nearest_init == np.arange(K),
        }
    )
    display(move_df[["displacement", "rel_displacement", "cos_sim", "cos_sim_centered"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
    print(f"avg pairwise distance between k-means centroids: {avg_dist_km:.1f}")
    print(f"avg pairwise distance between MFA means:         {avg_dist_mfa:.1f}")
    print(f"avg pairwise cosine between k-means centroids:   raw {avg_cos_km:.4f} | centered {avg_cos_km_centered:.4f}")
    print(f"avg pairwise cosine between MFA means:           raw {avg_cos_mfa:.4f} | centered {avg_cos_mfa_centered:.4f}")
    print(f"fraction of trained means still nearest their own init centroid: {frac_nearest_own_init:.3f}")

    if px is not None:
        fig = px.histogram(
            move_df,
            x="displacement",
            nbins=60,
            title="Centroid movement ‖mu_k − c_k‖ vs average inter-centroid distance",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=avg_dist_km, line_dash="dash", line_color=COLOR_KMEANS,
            annotation_text=f"avg dist between k-means centroids ({avg_dist_km:.1f})",
            annotation_position="top left", annotation_font_color=COLOR_KMEANS,
        )
        fig.add_vline(
            x=avg_dist_mfa, line_dash="dot", line_color=COLOR_MFA,
            annotation_text=f"avg dist between MFA means ({avg_dist_mfa:.1f})",
            annotation_position="bottom right", annotation_font_color=COLOR_MFA,
        )
        fig.update_layout(xaxis_title="displacement ‖mu_k − c_k‖", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "displacement_hist")
        fig.show()

        fig = px.histogram(
            move_df,
            x="rel_displacement",
            nbins=60,
            title="Centroid movement relative to local init spacing (‖mu_k − c_k‖ / nearest-init distance)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(x=1.0, line_dash="dash", line_color="#0b0b0b", annotation_text="moved past nearest init centroid")
        fig.update_layout(xaxis_title="relative displacement", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "rel_displacement_hist")
        fig.show()

        raw_start = float(np.floor(move_df["cos_sim"].min() * 100) / 100)
        fig = px.histogram(
            move_df,
            x="cos_sim",
            title="RAW cosine similarity between trained mean and init centroid (inflated by shared offset)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=avg_cos_km, line_dash="dash", line_color=COLOR_KMEANS,
            annotation_text=f"avg cos between k-means centroids ({avg_cos_km:.3f})",
            annotation_position="top left", annotation_font_color=COLOR_KMEANS,
        )
        fig.add_vline(
            x=avg_cos_mfa, line_dash="dot", line_color=COLOR_MFA,
            annotation_text=f"avg cos between MFA means ({avg_cos_mfa:.3f})",
            annotation_position="bottom left", annotation_font_color=COLOR_MFA,
        )
        fig.update_traces(xbins=dict(start=raw_start, end=1.0, size=(1.0 - raw_start) / 60))
        fig.update_layout(xaxis_title="cos(mu_k, c_k)", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "cos_raw_hist")
        fig.show()

        fig = px.histogram(
            move_df,
            x="cos_sim_centered",
            title="CENTERED cosine similarity between trained mean and init centroid (global mean removed)",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=avg_cos_km_centered, line_dash="dash", line_color=COLOR_KMEANS,
            annotation_text=f"avg centered cos between k-means centroids ({avg_cos_km_centered:.3f})",
            annotation_position="top left", annotation_font_color=COLOR_KMEANS,
        )
        fig.add_vline(
            x=avg_cos_mfa_centered, line_dash="dot", line_color=COLOR_MFA,
            annotation_text=f"avg centered cos between MFA means ({avg_cos_mfa_centered:.3f})",
            annotation_position="bottom left", annotation_font_color=COLOR_MFA,
        )
        fig.update_traces(xbins=dict(start=-1.0, end=1.0, size=0.025))
        fig.update_layout(xaxis_title="cos(mu_k − m, c_k − m)", yaxis_title="clusters", bargap=0.05, xaxis_range=[-1, 1])
        save_fig(fig, "cos_centered_hist")
        fig.show()


,displacement,rel_displacement,cos_sim,cos_sim_centered
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,14.251780,0.738935,0.994388,0.883078
std,6.375278,0.438777,0.004233,0.125482
min,0.794617,0.039332,0.967178,-0.229114
10%,6.085452,0.236935,0.988775,0.731359
25%,9.641117,0.405021,0.992469,0.832957
50%,13.887157,0.646062,0.995142,0.923085
75%,18.285810,1.002318,0.997408,0.974469
90%,22.595827,1.345213,0.998952,0.992119
max,45.445065,2.517992,0.999973,0.999856


avg pairwise distance between k-means centroids: 49.0
avg pairwise distance between MFA means:         52.7
avg pairwise cosine between k-means centroids:   raw 0.9471 | centered 0.0060
avg pairwise cosine between MFA means:           raw 0.9356 | centered 0.0018
fraction of trained means still nearest their own init centroid: 0.916


### 3.1 Neighbour Overlap: Init Centroid Cloud vs Trained Mean Cloud

Displacement and cosine track each centroid in isolation. Neighbour overlap
(`return_data_overlap` from `dalg.analysis.overlap`) instead treats the K init
centroids and the K trained means as two point clouds with row correspondence and
asks, for each cluster, what fraction of its k nearest *other centroids* is the same
in both clouds. Overlap 1 at all k means training moved the means rigidly enough to
keep the relational layout intact; overlap near the random baseline `k/(K−1)` means
each mean's surroundings were reshuffled even if the mean itself stayed close to its
init (and vice versa: large displacements can still preserve overlap if neighbours
moved together).


In [8]:
if not HAVE_MODEL:
    display(Markdown("**Skipped:** missing MFA model files or k-means centroids."))
else:
    from dalg.analysis.overlap import return_data_overlap

    # mu and c_init come from the Section 3 cell above
    NEIGHBOUR_KS = [1, 10, 25, 50, 100, 500]
    centroid_overlap = pd.DataFrame(
        {
            "k": NEIGHBOUR_KS,
            "neighbour_overlap": [return_data_overlap(c_init, mu, k=k) for k in NEIGHBOUR_KS],
        }
    )
    centroid_overlap["random_baseline"] = [k / (K - 1) for k in NEIGHBOUR_KS]
    display(centroid_overlap)


/u/dssc/zenocosini/decomposing-activations-local-geometry/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E0714 18:10:11.249272 1868260 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]


,k,neighbour_overlap,random_baseline
0,1,0.300000,0.001001
1,10,0.454800,0.010010
2,25,0.503200,0.025025
3,50,0.547280,0.050050
4,100,0.602940,0.100100
5,500,0.844262,0.500501


## 4. Intrinsic Dimension: Init Clusters vs MFA Clusters

**Experiment.** For each partition, `dalg-run-metrics intrinsic-dim` sampled up
to `max_samples` activations per cluster *according to that partition's own
assignments*, ran PCA on the centered samples, and recorded the number of
principal directions needed to reach the variance threshold (typically 90%) of
within-cluster variance. So each file characterizes the local geometry of its
own partition:

- **k-means IDs** describe the Voronoi regions around the init centroids;
- **MFA IDs** describe the regions after training reshaped the boundaries.

We compare (a) the two per-cluster ID **distributions**, and (b) the
**per-cluster change** `ΔID_k = ID_mfa[k] − ID_km[k]` under the identity
correspondence — including whether clusters whose *membership* changed most
(low Jaccard, Section 2) are also the ones whose *dimensionality* changed most.

Clusters with `intrinsic_dim == 0` were skipped by the metric (population below
`min_population`) and are masked out of per-cluster comparisons.

In [9]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    id_km_obj = torch.load(KMEANS_ID_PATH, map_location="cpu")
    id_mfa_obj = torch.load(MFA_ID_PATH, map_location="cpu")

    assert int(id_km_obj["K"]) == int(id_mfa_obj["K"]) == K, (id_km_obj["K"], id_mfa_obj["K"], K)
    assert id_km_obj["variance_threshold"] == id_mfa_obj["variance_threshold"], (
        id_km_obj["variance_threshold"],
        id_mfa_obj["variance_threshold"],
    )
    VAR_THRESHOLD = float(id_km_obj["variance_threshold"])

    id_km = id_km_obj["intrinsic_dims"].to(torch.long).numpy()
    id_mfa = id_mfa_obj["intrinsic_dims"].to(torch.long).numpy()
    valid = (id_km > 0) & (id_mfa > 0)
    print(
        f"variance threshold: {VAR_THRESHOLD}, "
        f"model_kind: kmeans={id_km_obj.get('model_kind')}, mfa={id_mfa_obj.get('model_kind')}"
    )
    print(f"clusters with valid ID in both: {int(valid.sum())}/{K}")

    id_summary = pd.DataFrame(
        [
            {
                "partition": name,
                "mean": vals.mean(),
                "median": np.median(vals),
                "p10": np.quantile(vals, 0.1),
                "p90": np.quantile(vals, 0.9),
                "min": vals.min(),
                "max": vals.max(),
            }
            for name, vals in [("kmeans (init)", id_km[valid]), ("mfa (trained)", id_mfa[valid])]
        ]
    )
    display(id_summary)

    if px is not None:
        id_long = pd.DataFrame(
            {
                "intrinsic_dim": np.concatenate([id_km[valid], id_mfa[valid]]),
                "partition": ["kmeans (init)"] * int(valid.sum()) + ["mfa (trained)"] * int(valid.sum()),
            }
        )
        fig = px.histogram(
            id_long,
            x="intrinsic_dim",
            color="partition",
            nbins=60,
            barmode="overlay",
            opacity=0.6,
            title=f"Per-cluster intrinsic dimension ({VAR_THRESHOLD:.0%} variance threshold)",
            color_discrete_map={"kmeans (init)": COLOR_KMEANS, "mfa (trained)": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title="intrinsic dimension", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "intrinsic_dim_hist")
        fig.show()

variance threshold: 0.9, model_kind: kmeans=assignments, mfa=mfa
clusters with valid ID in both: 987/1000


,partition,mean,median,p10,p90,min,max
0,kmeans (init),245.800405,237.0,129.6,378.8,12,582
1,mfa (trained),253.988855,257.0,131.0,367.6,12,670


### 4.1 Per-Cluster Change: Init → Trained

Same cluster id on both axes. Points below the diagonal are clusters whose
intrinsic dimension *dropped* during training.

In [10]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    id_df = pd.DataFrame(
        {
            "cluster": np.arange(K)[valid],
            "id_kmeans": id_km[valid],
            "id_mfa": id_mfa[valid],
            "km_cluster_size": id_km_obj["cluster_sizes"].numpy()[valid],
        }
    )
    id_df["delta_id"] = id_df["id_mfa"] - id_df["id_kmeans"]
    display(id_df["delta_id"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

    if px is not None:
        lim = float(max(id_df["id_kmeans"].max(), id_df["id_mfa"].max())) * 1.05
        fig = px.scatter(
            id_df,
            x="id_kmeans",
            y="id_mfa",
            hover_data=["cluster", "delta_id", "km_cluster_size"],
            title="Per-cluster intrinsic dimension: init vs trained",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.55,
        )
        fig.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim, line=dict(color="#0b0b0b", dash="dash", width=1))
        fig.update_layout(
            xaxis_title="intrinsic dim (k-means init partition)",
            yaxis_title="intrinsic dim (trained MFA partition)",
            xaxis_range=[0, lim],
            yaxis_range=[0, lim],
        )
        save_fig(fig, "id_init_vs_trained")
        fig.show()

        fig = px.histogram(
            id_df,
            x="delta_id",
            nbins=60,
            title="ΔID = ID(mfa) − ID(kmeans) per cluster",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(x=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_layout(xaxis_title="ΔID (trained − init)", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "delta_id_hist")
        fig.show()

        fig = px.scatter(
            id_df,
            x="km_cluster_size",
            y="delta_id",
            log_x=True,
            hover_data=["cluster", "id_kmeans", "id_mfa"],
            title="ΔID vs init cluster size",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
            opacity=0.55,
        )
        fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_layout(xaxis_title="k-means cluster size (tokens, log)", yaxis_title="ΔID")
        save_fig(fig, "delta_id_vs_cluster_size")
        fig.show()

count    987.000000
mean       8.188450
std       77.319107
min     -314.000000
10%      -90.000000
25%      -32.500000
50%       23.000000
75%       59.000000
90%       86.000000
max      367.000000
Name: delta_id, dtype: float64

### 4.2 Does Membership Change Predict Dimensionality Change?

Clusters whose token membership training reshaped the most (low Jaccard) might
also be the ones whose measured intrinsic dimension changed most. Requires both
the assignment (Section 2) and intrinsic-dim (Section 4) artifacts.

## 5. Spectral Profiles of the Two Partitions

**Experiment.** The intrinsic-dim artifacts store the full PCA variance
spectrum of every cluster (`cluster_variances`), so this section costs no new
computation. Section 4 showed the scalar ID (rank to 90% variance) drops from
~381 to ~203; here we look at the *whole spectrum* to see how:

- **Cumulative variance curves**: median and 10–90% band of the normalized
  cumulative spectrum across clusters, per partition. A partition whose curve
  rises faster has regions concentrated in fewer directions everywhere along
  the spectrum, not just at the 90% threshold.
- **Variance fraction in the top q directions** (q = the MFA rank): per-cluster
  distribution. Directly comparable to what a rank-q factor model can capture:
  if MFA regions concentrate much more variance in their top q PCs, the learned
  covariance has an easier job describing its own regions.
- **Spectral participation ratio** `(Σλ)² / Σλ²`: a threshold-free effective
  dimensionality of each cluster's spectrum.


In [11]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    RANK_GRID = np.arange(1, 1025)

    def spectrum_stats(obj):
        curves, topq, pr = [], [], []
        for k in range(K):
            if not valid[k]:
                continue
            v = obj["cluster_variances"][k].float().numpy()
            if v.size == 0 or v.sum() <= 0:
                continue
            cum = np.cumsum(v) / v.sum()
            curves.append(np.interp(RANK_GRID, np.arange(1, v.size + 1), cum))
            topq.append(cum[min(Q, v.size) - 1])
            pr.append(v.sum() ** 2 / (v ** 2).sum())
        return np.stack(curves), np.asarray(topq), np.asarray(pr)

    curves_km, topq_frac_km, pr_km = spectrum_stats(id_km_obj)
    curves_mfa, topq_frac_mfa, pr_mfa = spectrum_stats(id_mfa_obj)

    def rank_to(curves, thr):
        return (curves < thr).sum(axis=1) + 1

    spectral_summary = pd.DataFrame(
        [
            {
                "partition": name,
                f"median variance frac in top {Q} PCs": float(np.median(tq)),
                "median spectral participation ratio": float(np.median(pr_)),
                "median rank to 50% var": float(np.median(rank_to(cv, 0.5))),
                "median rank to 90% var": float(np.median(rank_to(cv, 0.9))),
            }
            for name, cv, tq, pr_ in [
                ("kmeans (init)", curves_km, topq_frac_km, pr_km),
                ("mfa (trained)", curves_mfa, topq_frac_mfa, pr_mfa),
            ]
        ]
    )
    display(spectral_summary)

    if go is not None:
        fig = go.Figure()
        for name, cv, color, fill in [
            ("kmeans (init)", curves_km, COLOR_KMEANS, "rgba(42,120,214,0.15)"),
            ("mfa (trained)", curves_mfa, COLOR_MFA, "rgba(27,175,122,0.15)"),
        ]:
            med = np.median(cv, axis=0)
            p10 = np.quantile(cv, 0.1, axis=0)
            p90 = np.quantile(cv, 0.9, axis=0)
            fig.add_trace(go.Scatter(x=RANK_GRID, y=p90, line=dict(width=0), showlegend=False, hoverinfo="skip"))
            fig.add_trace(go.Scatter(x=RANK_GRID, y=p10, fill="tonexty", fillcolor=fill, line=dict(width=0), showlegend=False, hoverinfo="skip"))
            fig.add_trace(go.Scatter(x=RANK_GRID, y=med, name=name, line=dict(color=color, width=2)))
        fig.update_layout(
            template=PLOT_TEMPLATE,
            title="Normalized cumulative variance spectrum per cluster (median, 10–90% band)",
            xaxis_title="PCA rank (log)",
            yaxis_title="cumulative variance fraction",
            xaxis_type="log",
        )
        fig.add_hline(y=0.9, line_dash="dash", line_color="#0b0b0b")
        save_fig(fig, "spectral_cumvar_curves")
        fig.show()

    if px is not None:
        topq_long = pd.DataFrame(
            {
                "topq_fraction": np.concatenate([topq_frac_km, topq_frac_mfa]),
                "partition": ["kmeans (init)"] * len(topq_frac_km) + ["mfa (trained)"] * len(topq_frac_mfa),
            }
        )
        fig = px.histogram(
            topq_long,
            x="topq_fraction",
            color="partition",
            nbins=60,
            barmode="overlay",
            opacity=0.6,
            title=f"Variance fraction captured by the top q={Q} PCs of each cluster",
            color_discrete_map={"kmeans (init)": COLOR_KMEANS, "mfa (trained)": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title=f"variance fraction in top {Q} PCs", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "spectral_topq_fraction_hist")
        fig.show()

        pr_long = pd.DataFrame(
            {
                "participation_ratio": np.concatenate([pr_km, pr_mfa]),
                "partition": ["kmeans (init)"] * len(pr_km) + ["mfa (trained)"] * len(pr_mfa),
            }
        )
        fig = px.histogram(
            pr_long,
            x="participation_ratio",
            color="partition",
            nbins=60,
            barmode="overlay",
            opacity=0.6,
            title="Spectral participation ratio per cluster (threshold-free effective dim)",
            color_discrete_map={"kmeans (init)": COLOR_KMEANS, "mfa (trained)": COLOR_MFA},
            template=PLOT_TEMPLATE,
        )
        fig.update_layout(xaxis_title="participation ratio of variance spectrum", yaxis_title="clusters", bargap=0.05)
        save_fig(fig, "spectral_pr_hist")
        fig.show()


,partition,median variance frac in top 10 PCs,median spectral participation ratio,median rank to 50% var,median rank to 90% var
0,kmeans (init),0.438979,32.369946,15.0,237.0
1,mfa (trained),0.510427,23.208006,10.0,257.0


### 5.1 Additional Top-PC Variance Fractions

The same per-cluster comparison as the top-`q=10` histogram above, repeated
for the top 50 and top 100 principal components.


In [12]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    EXTRA_TOPQ = (50, 100)

    def variance_fraction_at_ranks(obj, ranks):
        fractions = {rank: [] for rank in ranks}
        for cluster in range(K):
            if not valid[cluster]:
                continue
            variances = obj["cluster_variances"][cluster].double().numpy()
            variance_sum = variances.sum()
            if variances.size == 0 or variance_sum <= 0:
                continue
            cumulative = np.cumsum(variances) / variance_sum
            for rank in ranks:
                fractions[rank].append(cumulative[min(rank, variances.size) - 1])
        return {rank: np.asarray(values) for rank, values in fractions.items()}

    extra_topq_km = variance_fraction_at_ranks(id_km_obj, EXTRA_TOPQ)
    extra_topq_mfa = variance_fraction_at_ranks(id_mfa_obj, EXTRA_TOPQ)

    if px is not None:
        for q_plot in EXTRA_TOPQ:
            topq_long = pd.DataFrame(
                {
                    "topq_fraction": np.concatenate(
                        [extra_topq_km[q_plot], extra_topq_mfa[q_plot]]
                    ),
                    "partition": (
                        ["kmeans (init)"] * len(extra_topq_km[q_plot])
                        + ["mfa (trained)"] * len(extra_topq_mfa[q_plot])
                    ),
                }
            )
            fig = px.histogram(
                topq_long,
                x="topq_fraction",
                color="partition",
                nbins=60,
                barmode="overlay",
                opacity=0.6,
                title=f"Variance fraction captured by the top q={q_plot} PCs of each cluster",
                color_discrete_map={
                    "kmeans (init)": COLOR_KMEANS,
                    "mfa (trained)": COLOR_MFA,
                },
                template=PLOT_TEMPLATE,
            )
            fig.update_layout(
                xaxis_title=f"variance fraction in top {q_plot} PCs",
                yaxis_title="clusters",
                bargap=0.05,
            )
            save_fig(fig, f"spectral_top{q_plot}_fraction_hist")
            fig.show()


### 5.2 Per-Cluster Isotropy Change: Init → Trained

For each cluster, isotropy is the participation ratio normalized by its
finite-sample isotropic-null value:

`isotropy = (PR - 1) / (D(n - 1)/(D + n) - 1)`.

This normalization is important because the two PCA artifacts may use
different sample caps. Zero is the rank-one limit and one is the expected
value for an isotropic sample in `D` dimensions. The histogram shows the
distribution across matched clusters of
`Δ isotropy = isotropy_mfa - isotropy_kmeans`: positive values mean training
made a cluster more isotropic; negative values mean its variance became more
directionally concentrated.


In [13]:
if not HAVE_IDS:
    display(Markdown("**Skipped:** missing one or both intrinsic-dim artifacts."))
else:
    # The k-means artifact can be legacy and omit D, so recover the shared
    # ambient dimension from either artifact or, as a fallback, the centroids.
    _recorded_dims = {
        int(obj["D"])
        for obj in (id_km_obj, id_mfa_obj)
        if obj.get("D") is not None
    }
    if len(_recorded_dims) > 1:
        raise ValueError(f"Intrinsic-dim artifacts disagree on D: {_recorded_dims}")
    if _recorded_dims:
        ISOTROPY_D = _recorded_dims.pop()
    elif KMEANS_CENTROIDS.exists():
        ISOTROPY_D = int(_centroid_tensor(KMEANS_CENTROIDS).shape[1])
    else:
        raise ValueError("Cannot determine ambient dimension D for isotropy normalization")

    def cluster_isotropies(obj, ambient_dim):
        result = np.full(K, np.nan, dtype=float)
        sample_sizes = obj.get("sample_sizes")
        for cluster in range(K):
            if not valid[cluster]:
                continue
            variances = obj["cluster_variances"][cluster].double()
            variance_sum = variances.sum()
            if variances.numel() == 0 or variance_sum <= 0:
                continue
            n = (
                int(sample_sizes[cluster])
                if sample_sizes is not None
                else min(int(obj["cluster_sizes"][cluster]), int(obj["max_samples"]))
            )
            isotropic_null_pr = ambient_dim * (n - 1) / (ambient_dim + n)
            if isotropic_null_pr <= 1:
                continue
            participation_ratio = float(variance_sum.square() / variances.square().sum())
            result[cluster] = (participation_ratio - 1) / (isotropic_null_pr - 1)
        return result

    isotropy_km = cluster_isotropies(id_km_obj, ISOTROPY_D)
    isotropy_mfa = cluster_isotropies(id_mfa_obj, ISOTROPY_D)
    isotropy_valid = np.isfinite(isotropy_km) & np.isfinite(isotropy_mfa)
    isotropy_df = pd.DataFrame(
        {
            "cluster": np.arange(K)[isotropy_valid],
            "isotropy_kmeans": isotropy_km[isotropy_valid],
            "isotropy_mfa": isotropy_mfa[isotropy_valid],
        }
    )
    isotropy_df["delta_isotropy"] = (
        isotropy_df["isotropy_mfa"] - isotropy_df["isotropy_kmeans"]
    )
    print(
        f"clusters with isotropy in both partitions: {len(isotropy_df)}/{K}; "
        f"ambient D={ISOTROPY_D}"
    )
    display(isotropy_df["delta_isotropy"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

    if px is not None:
        fig = px.histogram(
            isotropy_df,
            x="delta_isotropy",
            nbins=60,
            title="Per-cluster isotropy change after MFA training",
            color_discrete_sequence=[COLOR_NEUTRAL],
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(x=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_layout(
            xaxis_title="Δ isotropy (MFA trained − k-means init)",
            yaxis_title="clusters",
            bargap=0.05,
        )
        save_fig(fig, "delta_isotropy_hist")
        fig.show()


clusters with isotropy in both partitions: 987/1000; ambient D=2048


count    987.000000
mean      -0.035266
std        0.047874
min       -0.264138
10%       -0.103251
25%       -0.036066
50%       -0.016921
75%       -0.008293
90%       -0.003254
max        0.019787
Name: delta_isotropy, dtype: float64

## 7. Where Do the Learned Subspaces W_k Point?

**Experiment.** Each MFA component owns a rank-q loading matrix `W_k` whose
span is its local subspace. We orthonormalize it and compare it with the cached
top-q empirical PCs of the same MFA region. The reference is the
**random-subspace baseline q/D**: a random unit vector has expected squared
projection q/D onto a random q-dimensional subspace.


**Empirical-PC alignment**: compare `span(W_k)` with the cached top-q PCs
of its own MFA region using the mean squared cosine of the principal angles.


For orthonormal bases `Q_i, U_j ∈ R^{D×q}`, the singular values of
`Q_iᵀ U_j` are `cos θ₁, …, cos θ_q`, so
`overlap(i,j) = ‖Q_iᵀ U_j‖²_F / q = mean_l cos² θ_l`. We compare each
component with its own empirical region and with a deterministic wrong-region
pairing as a control.


In [14]:
subspace_df = pd.DataFrame()
subspace_summary = pd.DataFrame()

_have_section7_model = (MFA_RUN / "mfa_model.pt").exists() or (MFA_RUN / "mfa_model_shards.json").exists()
if not (_have_section7_model and HAVE_TOP_PCS and pcs_mfa is not None):
    display(Markdown("**Skipped:** requires the MFA model and MFA cluster_top_pcs.pt."))
else:
    from dalg.models.mfa import load_mfa as _load_mfa_section7

    # Positive loading scales and the optional invertible factor rotation do
    # not change the column span. QR on dir_raw therefore recovers span(W_k)
    # without constructing another full (K, D, q) loading tensor.
    _model7 = _load_mfa_section7(MFA_RUN / "mfa_model.pt", map_location="cpu")
    assert _model7.K == K and _model7.q == Q
    _q_chunks = []
    with torch.no_grad():
        for _start in range(0, K, 64):
            _dirs = _model7.dir_raw[_start : _start + 64].detach().float()
            _q_chunks.append(torch.linalg.qr(_dirs, mode="reduced")[0].cpu())
    Q_bases = torch.cat(_q_chunks, dim=0)
    del _model7, _q_chunks, _dirs
    gc.collect()

    SUBSPACE_RANK = min(Q, TOP_PCS)
    PCA_D = int(Q_bases.shape[1])
    RANDOM_SUBSPACE_BASELINE = SUBSPACE_RANK / PCA_D
    section7_ids = np.asarray(
        [k for k, basis in enumerate(pcs_mfa) if basis is not None and basis.shape[0] >= SUBSPACE_RANK],
        dtype=int,
    )
    if len(section7_ids) < 2:
        raise RuntimeError("Section 7 needs empirical PCA bases for at least two MFA clusters")

    # PCA components are stored row-wise; transpose the first q rows to D x q.
    U_bases = torch.stack(
        [pcs_mfa[int(k)][:SUBSPACE_RANK].T.float() for k in section7_ids], dim=0
    )
    Q_eval = Q_bases[torch.from_numpy(section7_ids), :, :SUBSPACE_RANK]
    _eye = torch.eye(SUBSPACE_RANK)
    raw_pc_basis_max_orth_error = float(
        (torch.matmul(U_bases.transpose(1, 2), U_bases) - _eye).abs().max()
    )
    # The PCA sidecar is numerically close to orthonormal but not exact. QR
    # makes the principal-angle interpretation exact without changing its span.
    U_bases = torch.linalg.qr(U_bases, mode="reduced")[0]
    pc_basis_max_orth_error = float(
        (torch.matmul(U_bases.transpose(1, 2), U_bases) - _eye).abs().max()
    )
    assert pc_basis_max_orth_error < 5e-4, pc_basis_max_orth_error

    same_cross = torch.einsum("kdq,kdr->kqr", Q_eval, U_bases)
    same_cosines = torch.linalg.svdvals(same_cross).clamp_(0.0, 1.0)
    same_overlap = same_cosines.square().mean(dim=1).numpy()
    median_angle_deg = torch.rad2deg(torch.acos(same_cosines)).median(dim=1).values.numpy()

    # One deterministic derangement: every Q_k is paired with another valid U_j.
    _rng7 = np.random.default_rng(0)
    _order = _rng7.permutation(len(section7_ids))
    wrong_index = np.empty(len(section7_ids), dtype=int)
    wrong_index[_order] = np.roll(_order, -1)
    assert np.all(wrong_index != np.arange(len(section7_ids)))
    wrong_cross = torch.einsum("kdq,kdr->kqr", Q_eval, U_bases[wrong_index])
    wrong_overlap = torch.linalg.svdvals(wrong_cross).square().mean(dim=1).numpy()

    subspace_df = pd.DataFrame(
        {
            "cluster": section7_ids,
            "overlap": same_overlap,
            "median_principal_angle_deg": median_angle_deg,
            "wrong_cluster": section7_ids[wrong_index],
            "wrong_cluster_overlap": wrong_overlap,
        }
    )

    def _section7_summary_row(metric: str, values: np.ndarray) -> dict:
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        return {
            "metric": metric,
            "n": len(values),
            "mean": float(np.mean(values)),
            "p10": float(np.quantile(values, 0.1)),
            "median": float(np.median(values)),
            "p90": float(np.quantile(values, 0.9)),
        }

    subspace_summary = pd.DataFrame(
        [
            _section7_summary_row("span(W_k) vs own MFA-region top-q PCs", same_overlap),
            _section7_summary_row("span(W_k) vs wrong MFA-region top-q PCs", wrong_overlap),
            _section7_summary_row("median principal angle (degrees)", median_angle_deg),
        ]
    )
    display(subspace_summary)
    print(
        f"valid MFA PCA clusters: {len(section7_ids)}/{K}; rank={SUBSPACE_RANK}; D={PCA_D}; "
        f"random-subspace overlap q/D={RANDOM_SUBSPACE_BASELINE:.6f}; "
        f"PCA orthonormality error raw={raw_pc_basis_max_orth_error:.2e}, after QR={pc_basis_max_orth_error:.2e}"
    )
    print(f"PCA assignment provenance: {top_pcs_mfa_obj.get('assignments_path', 'not recorded')}")

    if px is not None:
        _overlap_long = pd.DataFrame(
            {
                "overlap": np.concatenate([same_overlap, wrong_overlap]),
                "pairing": ["own MFA region"] * len(same_overlap) + ["wrong MFA region"] * len(wrong_overlap),
            }
        )
        fig = px.histogram(
            _overlap_long, x="overlap", color="pairing", nbins=60, barmode="overlay", opacity=0.65,
            title=f"Learned rank-{SUBSPACE_RANK} subspace vs empirical MFA-region PCs",
            color_discrete_map={"own MFA region": COLOR_MFA, "wrong MFA region": COLOR_NEUTRAL},
            template=PLOT_TEMPLATE,
        )
        fig.add_vline(
            x=RANDOM_SUBSPACE_BASELINE, line_dash="dash", line_color="#0b0b0b",
            annotation_text=f"random q/D ({RANDOM_SUBSPACE_BASELINE:.4f})", annotation_position="top right",
        )
        fig.update_layout(xaxis_title="mean cos² of principal angles", yaxis_title="cluster pairs", bargap=0.05)
        save_fig(fig, "w_empirical_pc_overlap_hist")
        fig.show()
    del Q_bases, Q_eval, U_bases, same_cross, same_cosines, wrong_cross
    gc.collect()


,metric,n,mean,p10,median,p90
0,span(W_k) vs own MFA-region top-q PCs,987,0.845822,0.778449,0.856894,0.911510
1,span(W_k) vs wrong MFA-region top-q PCs,987,0.081634,0.045471,0.080093,0.114898
2,median principal angle (degrees),987,7.477536,4.384015,6.864557,10.805763


valid MFA PCA clusters: 987/1000; rank=10; D=2048; random-subspace overlap q/D=0.004883; PCA orthonormality error raw=1.07e-03, after QR=1.55e-06
PCA assignment provenance: /orfeo/scratch/dssc/zenocosini/dalg-cache/pile_gemma2b_models/layer05_1000_10_component_sharded_mfa/mfa_model_assignments.pt


## 8. Per-Cluster PCA Subspace Alignment: K-Means Regions vs MFA Regions

This section asks whether assignment changes also reshape the local geometry of a
cluster. The PCA sidecars loaded in Section 1 contain the top 100 empirical PCs
computed independently from the k-means and MFA partitions. For every cluster with
valid PCs in both, and for each rank in `PA_KS`, we quantify the alignment of the
two k-dimensional subspaces with **principal angles**: the singular values of
subspaces with **principal angles**: the singular values of `Q_kmᵀ Q_mfa` are
`cos θ₁ ≥ … ≥ cos θ_k`, and we report

- `mean cos²θ = ‖Q_kmᵀ Q_mfa‖²_F / k` — 1 when the spans coincide, expectation `k/D`
  (≈ 0.005–0.05 here) for random k-dim subspaces in D = 2048;
- the median principal angle in degrees — 0° for identical spans, → 90° for random ones.

The cached bases are re-orthonormalized through their Gram matrices before the
angles are computed. The overlap histogram shows the full distribution across
matched clusters separately for every PCA rank.


In [19]:
pca_alignment_df = pd.DataFrame()
pca_alignment_summary = pd.DataFrame()

if not HAVE_TOP_PCS:
    display(Markdown("**Skipped:** requires both k-means and MFA cluster_top_pcs.pt sidecars."))
else:
    PA_KS = [k for k in (10, 25, 50, 100) if k <= TOP_PCS]
    PA_BATCH_SIZE = 8
    pa_cluster_ids = np.flatnonzero(valid_pcs_both)
    if len(pa_cluster_ids) == 0:
        raise RuntimeError("Section 8 needs at least one cluster with PCs in both partitions")

    max_pa_rank = max(PA_KS)
    _eye_max = torch.eye(max_pa_rank)
    max_raw_pc_orth_error = 0.0
    _alignment_rows = []

    # Work in small batches. For row-wise PCA bases A and B, whiten their
    # Gram matrices before taking SVD of A B^T so the singular values are
    # the cosines of principal angles even when the cached rows are only
    # approximately orthonormal.
    for _start in range(0, len(pa_cluster_ids), PA_BATCH_SIZE):
        _batch_ids = pa_cluster_ids[_start : _start + PA_BATCH_SIZE]
        _a = torch.stack([pcs_km[int(k)][:max_pa_rank].float() for k in _batch_ids])
        _b = torch.stack([pcs_mfa[int(k)][:max_pa_rank].float() for k in _batch_ids])
        _gram_a = torch.matmul(_a, _a.transpose(1, 2))
        _gram_b = torch.matmul(_b, _b.transpose(1, 2))
        _cross_ab = torch.matmul(_a, _b.transpose(1, 2))
        max_raw_pc_orth_error = max(
            max_raw_pc_orth_error,
            float((_gram_a - _eye_max).abs().max()),
            float((_gram_b - _eye_max).abs().max()),
        )

        for _rank in PA_KS:
            _chol_a = torch.linalg.cholesky(_gram_a[:, :_rank, :_rank])
            _chol_b = torch.linalg.cholesky(_gram_b[:, :_rank, :_rank])
            _cross = torch.linalg.solve_triangular(
                _chol_a, _cross_ab[:, :_rank, :_rank], upper=False
            )
            _cross = torch.linalg.solve_triangular(
                _chol_b, _cross.transpose(1, 2), upper=False
            ).transpose(1, 2)
            _cosines = torch.linalg.svdvals(_cross).clamp_(0.0, 1.0)
            _overlap = _cosines.square().mean(dim=1)
            _median_angle = torch.quantile(
                torch.rad2deg(torch.acos(_cosines)), 0.5, dim=1
            )
            for _i, _cluster in enumerate(_batch_ids):
                _alignment_rows.append(
                    {
                        "cluster": int(_cluster),
                        "rank": _rank,
                        "overlap": float(_overlap[_i]),
                        "median_principal_angle_deg": float(_median_angle[_i]),
                    }
                )

        del _a, _b, _gram_a, _gram_b, _cross_ab, _chol_a, _chol_b, _cross, _cosines

    pca_alignment_df = pd.DataFrame(_alignment_rows)
    pca_alignment_df["random_baseline"] = pca_alignment_df["rank"] / PCA_D

    _summary_rows = []
    for _rank in PA_KS:
        _rank_df = pca_alignment_df[pca_alignment_df["rank"] == _rank]
        _summary_rows.append(
            {
                "rank": _rank,
                "clusters": len(_rank_df),
                "random_baseline": _rank / PCA_D,
                "mean_overlap": float(_rank_df["overlap"].mean()),
                "p10_overlap": float(_rank_df["overlap"].quantile(0.1)),
                "median_overlap": float(_rank_df["overlap"].median()),
                "p90_overlap": float(_rank_df["overlap"].quantile(0.9)),
                "median_angle_deg": float(_rank_df["median_principal_angle_deg"].median()),
            }
        )
    pca_alignment_summary = pd.DataFrame(_summary_rows)
    display(pca_alignment_summary)
    print(
        f"clusters with PCs in both partitions: {len(pa_cluster_ids)}/{K}; D={PCA_D}; "
        f"maximum raw PCA Gram error={max_raw_pc_orth_error:.2e}"
    )
    print(f"K-means PCA provenance: {top_pcs_km_obj.get('assignments_path', 'not recorded')}")
    print(f"MFA PCA provenance: {top_pcs_mfa_obj.get('assignments_path', 'not recorded')}")

    if px is not None:
        for _rank in PA_KS:
            _rank_df = pca_alignment_df[pca_alignment_df["rank"] == _rank]
            fig = px.histogram(
                _rank_df,
                x="overlap",
                nbins=50,
                title=f"Matched-cluster PCA subspace overlap at rank {_rank}",
                color_discrete_sequence=[COLOR_NEUTRAL],
                template=PLOT_TEMPLATE,
            )
            fig.update_layout(
                xaxis_title="mean cos² of principal angles (overlap)",
                yaxis_title="clusters",
                bargap=0.05,
            )
            fig.update_xaxes(range=[0, 1])
            save_fig(fig, f"kmeans_mfa_pca_overlap_hist_rank{_rank}")
            fig.show()

    del _alignment_rows
    gc.collect()


,rank,clusters,random_baseline,mean_overlap,p10_overlap,median_overlap,p90_overlap,median_angle_deg
0,10,979,0.004883,0.610230,0.328830,0.628663,0.846595,25.711264
1,25,979,0.012207,0.603560,0.357246,0.615052,0.827326,25.610985
2,50,979,0.024414,0.603400,0.375446,0.610243,0.821093,27.652302
3,100,979,0.048828,0.619997,0.424642,0.620724,0.812098,27.193855


clusters with PCs in both partitions: 979/1000; D=2048; maximum raw PCA Gram error=1.12e-03
K-means PCA provenance: /orfeo/scratch/dssc/zenocosini/dalg-cache/pile_gemma2b_models/centroids/k1000_L05/kmeans_centroid_assignments.pt
MFA PCA provenance: /orfeo/scratch/dssc/zenocosini/dalg-cache/pile_gemma2b_models/layer05_1000_10_component_sharded_mfa/mfa_model_assignments.pt


## 9. Compact Conclusion

In [16]:
rows = []
if HAVE_ASSIGNMENTS:
    rows += [
        {"analysis": "assignments", "metric": "same-id agreement", "value": same_id_agreement},
        {"analysis": "assignments", "metric": "NMI", "value": NMI},
        {"analysis": "assignments", "metric": "fraction of clusters self-matched (Hungarian)", "value": frac_self_matched},
        {"analysis": "assignments", "metric": "median per-cluster Jaccard", "value": float(per_cluster["jaccard"].median())},
    ]
if HAVE_MODEL:
    rows += [
        {"analysis": "centroid movement", "metric": "median relative displacement (vs nearest-init distance)", "value": float(np.median(rel_displacement))},
        {"analysis": "centroid movement", "metric": "median centered cos(mu_trained, c_init)", "value": float(np.median(cos_sim_centered))},
        {"analysis": "centroid movement", "metric": "avg centered cos between different k-means centroids", "value": avg_cos_km_centered},
        {"analysis": "centroid movement", "metric": "fraction of trained means nearest own init centroid", "value": frac_nearest_own_init},
    ]
if HAVE_IDS:
    rows += [
        {"analysis": "intrinsic dim", "metric": "mean ID kmeans (init)", "value": float(id_km[valid].mean())},
        {"analysis": "intrinsic dim", "metric": "mean ID mfa (trained)", "value": float(id_mfa[valid].mean())},
        {"analysis": "intrinsic dim", "metric": "median ΔID (trained − init)", "value": float(id_df["delta_id"].median())},
    ]
if HAVE_IDS and "topq_frac_km" in globals():
    rows += [
        {"analysis": "spectral", "metric": f"median variance frac in top {Q} PCs (kmeans regions)", "value": float(np.median(topq_frac_km))},
        {"analysis": "spectral", "metric": f"median variance frac in top {Q} PCs (mfa regions)", "value": float(np.median(topq_frac_mfa))},
    ]
if HAVE_MODEL and "subspace_df" in globals() and not subspace_df.empty:
    rows += [
        {"analysis": "subspace", "metric": "median overlap span(W) vs top-q empirical PCs", "value": float(subspace_df["overlap"].median())},
        {"analysis": "subspace", "metric": "median overlap span(W) vs wrong-cluster top-q PCs", "value": float(subspace_df["wrong_cluster_overlap"].median())},
        {"analysis": "subspace", "metric": "median principal angle span(W) vs own-region PCs (degrees)", "value": float(subspace_df["median_principal_angle_deg"].median())},
    ]
if "pca_alignment_summary" in globals() and not pca_alignment_summary.empty:
    _pa_q_row = pca_alignment_summary.loc[pca_alignment_summary["rank"] == min(Q, max(PA_KS))].iloc[0]
    _pa_max_row = pca_alignment_summary.loc[pca_alignment_summary["rank"] == max(PA_KS)].iloc[0]
    rows += [
        {"analysis": "partition PCA alignment", "metric": f"median overlap at rank {int(_pa_q_row['rank'])}", "value": float(_pa_q_row["median_overlap"])},
        {"analysis": "partition PCA alignment", "metric": f"median principal angle at rank {int(_pa_q_row['rank'])} (degrees)", "value": float(_pa_q_row["median_angle_deg"])},
        {"analysis": "partition PCA alignment", "metric": f"median overlap at rank {int(_pa_max_row['rank'])}", "value": float(_pa_max_row["median_overlap"])},
    ]
if HAVE_ASSIGNMENTS and HAVE_IDS and "merged" in globals():
    rows.append(
        {
            "analysis": "coupling",
            "metric": "Spearman(jaccard, |ΔID|)",
            "value": float(merged["jaccard"].corr(merged["abs_delta_id"], method="spearman")),
        }
    )
if rows:
    display(pd.DataFrame(rows))
else:
    display(Markdown("**Skipped:** no artifacts available."))

,analysis,metric,value
0,assignments,same-id agreement,0.573012
1,assignments,NMI,0.723901
2,assignments,fraction of clusters self-matched (Hungarian),0.992000
3,assignments,median per-cluster Jaccard,0.404690
4,centroid movement,median relative displacement (vs nearest-init ...,0.646062
5,centroid movement,"median centered cos(mu_trained, c_init)",0.923085
6,centroid movement,avg centered cos between different k-means cen...,0.006049
7,centroid movement,fraction of trained means nearest own init cen...,0.916000
8,intrinsic dim,mean ID kmeans (init),245.800405
9,intrinsic dim,mean ID mfa (trained),253.988855
